# CSC271: Web Scraping using BeautifulSoup

## Lecture Topics
- Scraping non-tabular HTML content
- The `BeautifulSoup` library

### Introduction

Last lecture, we used the pandas function `read_html` to extract data from HTML tables into pandas DataFrames. Unfortunately, not all HTML content that we want to extract appears within a table.

Let's consider this site, which contains a list of faculty and students who belong to the UofT Computer Science Education Research Group:

https://uoftcsed.github.io/people/

In [1]:
import pandas as pd
import requests
from io import StringIO

url = "https://uoftcsed.github.io/people/"

response = requests.get(url)
response.encoding = response.apparent_encoding

if response.status_code == 200:
    html = response.text
else:
    print(f'The request was not successful: status code {response.status_code}')

If we attempt to use `read_html` on the HTML from that site, a `ValueError` occurs because there are no tables on that webpage.

In [2]:
html = pd.read_html(StringIO(html))

ImportError: Missing optional dependency 'lxml'.  Use pip or conda to install lxml.

Consider how we could go about finding the URLs in the html text. We could use loops and the `str` method `find`. That approach would be very error-prone, since it would depend heavily on spacing and exact HTML formatting! You can see a version of that approach implemented below:

In [3]:
import requests

url = "https://uoftcsed.github.io/people/"
response = requests.get(url)
html = response.text

urls = []
pos = 0

while pos < len(html):

    a_start = html.find("<a", pos)

    if a_start != -1:
        a_end = html.find(">", a_start)

        if a_end != -1:
            tag = html[a_start:a_end]

            href_pos = tag.find('href=')

            if href_pos != -1:
                quote = tag[href_pos + len('href=')]
                start = href_pos + len('href=') + 1
                end = tag.find(quote, start)

                if end != -1:
                    urls.append(tag[start:end])

            pos = a_end + 1
        else:
            pos = len(html)
    else:
        pos = len(html)

print(urls)


['http://uoftcsed.github.io', 'http://uoftcsed.github.io/people', 'http://uoftcsed.github.io/pubs', 'http://uoftcsed.github.io/books', 'http://uoftcsed.github.io/projects', 'https://www.utm.utoronto.ca/math-cs-stats/people/andreas-andi-bergen', 'https://www.cs.toronto.edu/~calver/', 'https://jencampbell.github.io/', 'http://www.utsc.utoronto.ca/cms/nick-cheng', 'https://michellecraig.github.io/', 'http://www.cs.toronto.edu/~sme/', 'http://www.cs.toronto.edu/~sengels/', 'https://www.utm.utoronto.ca/math-cs-stats/people/rutwa-engineer', 'http://www.cs.utoronto.ca/~strider/', 'https://www.cs.toronto.edu/~axgao/', 'http://www.cs.toronto.edu/~pgries/', 'http://www.tovigrossman.com/', 'http://brianharrington.net/', 'http://www.cs.toronto.edu/~heap/', 'http://www.cs.toronto.edu/~dianeh/', 'http://www.cs.toronto.edu/~david/', 'https://www.michaelliut.ca/', 'http://andrewpetersen.info/', 'http://www.cs.toronto.edu/~fpitt/', 'http://www.cs.toronto.edu/~reid/', 'http://www.cs.toronto.edu/~arnold/

## BeautifulSoup

The BeautifulSoup library has functions for pulling data out of HTML. It can be used to find HTML elements by tag, class, or attributes. It can also extract links, text, tables, and other content.

<div class="alert alert-block alert-success">
<b>Installing <code>BeautifulSoup</code></b>

To install BeautifulSoup, open the Terminal in VS Code and run:

`pip install beautifulsoup4`
or 
`pip3 install beautifulsoup4`
</div>

We'll start by creating a BeautifulSoup object based on the HTML for the CS Ed webpage. The BeautifulSoup object represents the HTML page as a nested data structure. We can inspect the `soup` object's attributes and use its `get_text` method to extract text from within elements:

Let's consider this smaller HTML page.

```html
<html lang="en">
  <head>
    <title>Demo Webpage</title>
    <h2>Subtitle</h2>
  </head>
  <body>
    <h2>BODY</h2>
    <div class="site">
        <ul>
            <li>First</li>
            <li>Last</li>
        </ul>
    </div>
  </body>
</html>
```

Here is its tree structure:

```
html (lang="en")
├── head
│   ├── title: "Demo Webpage"
│   └── h2: "Subtitle"
└── body
    ├── h2: "BODY"
    └── div (class="site")
        └── ul
            ├── li: "First"
            └── li: "Last"
```
We use `BeautifulSoup` to create a tree structure where each element in the tree has a `Tag` type. We can then navigate the entire tree or branches of it.

In [5]:
from bs4 import BeautifulSoup

demo_html = '''<html lang="en">
  <head>
    <title>Demo Webpage</title>
    <h2>Subtitle</h2>
  </head>
  <body>
    <h2>BODY</h2>
    <div class="site">
        <ul>
            <li>First</li>
            <li>Last</li>
        </ul>
    </div>
  </body>
</html>'''

demo_soup = BeautifulSoup(demo_html, 'html.parser')

# Demo

# Subtrees and paths
print(demo_soup.head)
print(demo_soup.body)
print(demo_soup.body.div.ul.li)




<head>
<title>Demo Webpage</title>
<h2>Subtitle</h2>
</head>
<body>
<h2>BODY</h2>
<div class="site">
<ul>
<li>First</li>
<li>Last</li>
</ul>
</div>
</body>
<li>First</li>


In [6]:
# Get an attribute's value
print(demo_soup.html.get('lang'))

en


In [15]:
# find, find_next, find_previous, find_all
print(demo_soup.find('div', class_='site'))
print(demo_soup.find('div'))

print(demo_soup.find('h2'))
print(demo_soup.find_all('h2'))
h2_tags = demo_soup.find_all('h2')
print(h2_tags[0])
print(h2_tags[1])

print(demo_soup.html.body.div.ul.li.find_next())
print(demo_soup.html.body.div.ul.li.find_next().find_previous())

<div class="site">
<ul>
<li>First</li>
<li>Last</li>
</ul>
</div>
<div class="site">
<ul>
<li>First</li>
<li>Last</li>
</ul>
</div>
<h2>Subtitle</h2>
[<h2>Subtitle</h2>, <h2>BODY</h2>]
<h2>Subtitle</h2>
<h2>BODY</h2>
<li>Last</li>
<li>First</li>


In [20]:
# Working within subtrees
print(demo_soup.find_all('h2'))
body = demo_soup.find('body')
print(body.find_all('h2'))

[<h2>Subtitle</h2>, <h2>BODY</h2>]
[<h2>BODY</h2>]


In [23]:
# get_text
print(demo_soup.title.get_text())
print(h2_tags[0].get_text())
print(h2_tags[1].get_text())

Demo Webpage
Subtitle
BODY


<div class="alert alert-block alert-info">
<h3>BeautifulSoup Methods</h3>

<table>
  <thead>
    <tr>
      <th>Method</th>
      <th>Brief description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>soup.find(name, **attrs) -&gt; Tag | None</code></td>
      <td>
        Return the <strong>first</strong> matching element in the search scope as
        a <code>Tag</code>, or <code>None</code> if no match is found. Support
        filtering by HTML attributes (e.g., <code>id</code>,
        <code>class_</code>, <code>href</code>).
      </td>
    </tr>
    <tr>
      <td><code>soup.find_all(name, **attrs) -&gt; list[Tag]</code></td>
      <td>
        Return <strong>all</strong> matching elements as a list of
        <code>Tag</code> objects, or an empty list if no matches are found.
        Support filtering by HTML attributes.
      </td>
    </tr>
    <tr>
      <td><code>tag.get_text(strip=False, separator='') -&gt; str</code></td>
      <td>
        Return the text content of a tag, including all descendant text.
        Optionally remove surrounding whitespace (<code>strip=True</code>) and
        specify a separator for nested text.
      </td>
    </tr>
    <tr>
      <td><code>tag.get(attr) -&gt; str | None</code></td>
      <td>
        Return the value of the specified HTML attribute, or <code>None</code>
        if the attribute does not exist.
      </td>
    </tr>
    <tr>
      <td><code>tag.find(name, **attrs) -&gt; Tag | None</code></td>
      <td>
        Return the <strong>first</strong> matching descendant element as a
        <code>Tag</code>, or <code>None</code> if no match is found. Support
        filtering by HTML attributes.
      </td>
    </tr>
    <tr>
      <td><code>tag.find_all(name, **attrs) -&gt; list[Tag]</code></td>
      <td>
        Return <strong>all</strong> matching descendant elements as a list of
        <code>Tag</code> objects. Support filtering by HTML attributes.
      </td>
    </tr>
    <tr>
      <td>
        <code>tag.find_previous(name=None, **attrs) -&gt; Tag | None</code>
      </td>
      <td>
        Return the <strong>first</strong> matching element found by searching
        <strong>backward</strong> in the document, or <code>None</code> if no
        match is found. Support filtering by HTML attributes.
      </td>
    </tr>
    <tr>
      <td><code>tag.find_next -&gt; Tag | NavigableString | None</code></td>
      <td>
        Return the <strong>next sibling node</strong> in the document, which may
        be a <code>Tag</code>, a <code>NavigableString</code>, or
        <code>None</code> if no next sibling exists.
      </td>
    </tr>
  </tbody>
</table>

Beautiful Soup Documentation:
https://beautiful-soup-4.readthedocs.io/en/latest/
</div>

Let's revisit the task of getting all URLs from the CS Ed Research Group webpage. We can start by using BeautifulSoup's `find_all` method to get all links (`<a></a>`):

In [25]:
soup = BeautifulSoup(html, 'html.parser')

urls = []

for link in soup.find_all('a'):
    urls.append(link.get('href'))

print(urls)

['http://uoftcsed.github.io', 'http://uoftcsed.github.io/people', 'http://uoftcsed.github.io/pubs', 'http://uoftcsed.github.io/books', 'http://uoftcsed.github.io/projects', 'https://www.utm.utoronto.ca/math-cs-stats/people/andreas-andi-bergen', 'https://jencampbell.github.io/', 'http://www.utsc.utoronto.ca/cms/nick-cheng', 'https://michellecraig.github.io/', 'http://www.cs.toronto.edu/~sme/', 'http://www.cs.toronto.edu/~sengels/', 'http://www.cs.utoronto.ca/~strider/', 'https://www.cs.toronto.edu/~axgao/', 'http://www.cs.toronto.edu/~pgries/', 'http://www.tovigrossman.com/', 'http://brianharrington.net/', 'http://www.cs.toronto.edu/~heap/', 'http://www.cs.toronto.edu/~dianeh/', 'http://www.cs.toronto.edu/~david/', 'https://www.michaelliut.ca/', 'http://andrewpetersen.info/', 'http://www.cs.toronto.edu/~fpitt/', 'http://www.cs.toronto.edu/~reid/', 'http://www.cs.toronto.edu/~arnold/', 'http://rainsharmin.com/', 'https://www.cs.toronto.edu/~bogdan/', 'http://www.jesmith.ca/', 'http://www

Now let's work on extracting the categories (e.g., Faculty) and names of the people on this page. Here is the structure of a key section of the page. There are other `<h2>` and `<div>` tags on the page. These are the only `<div>` tags with a class attribute of `post`.

```html
<h2>Faculty</h2>
<div class="posts">
  <a href="...">NAME1</a>
  ...
</div>

<h2>Graduate Students</h2>
<div class="posts">
  <a href="...">NAME2</a>
  ...
</div>

<h2>Undergraduate students</h2>
<div class="posts">
  <a href="...">NAME3</a>
  ...
</div>
```

The goal is to produce a dictionary of the form:
```
{'Faculty': ['NAME1', ...], 
 'Graduate Students': ['NAME2', ...],
 'Undergraduate Students': ['NAME3', ...]}
```

We can write a program that uses BeautifulSoup to extract this information. 

<div class="alert alert-block alert-success">
Consider what strategy to take to extract this information. Which tags to look for? What to do next?
</div>

In [ ]:
soup = BeautifulSoup(html, "html.parser")

section_to_names = {}

for div in soup.find_all('div', class_='posts'):

    h2 = div.find_previous('h2')
    section_title = h2.get_text()
    section_to_names[section_title] = []

    for a in div.find_all('a'):
        name = a.get_text(' ', strip=True)
        section_to_names[section_title].append(name)

print(section_to_names)


{'Faculty': ['Andreas Bergen', 'Jennifer Campbell', 'Nick Cheng', 'Michelle Craig', 'Steve Easterbrook', 'Steve Engels', 'Francisco Estrada', 'Alice Gao', 'Paul Gries', 'Tovi Grossman', 'Brian Harrington', 'Danny Heap', 'Diane Horton', 'David Liu', 'Michael Liut', 'Andrew Petersen', 'François Pitt', 'Karen Reid', 'Arnold Rosenbloom', 'Sadia Sharmin', 'Bogdan Simion', 'Jacqueline Smith', 'Anya Tafliovich', 'Joseph Jay Williams', 'Lisa Zhang', 'Tingting Zhu', 'Daniel Zingaro'], 'Graduate Students': ['Majeed Kazemitabaar', 'Suqing (Richard) Liu', 'Naaz Sibia', 'Harry Ye'], 'Undergraduate Students': ['Yashika Jain', 'Khushi Malik', 'Vaidik Patel', 'Caroline Pechenik', 'Adam Raway', 'Amber Richardson', 'Ishan Singh', 'Jessica Wen', 'Mandy Zhou']}


Although we can use the `pandas` function `read_html` to extract entire tables, `BeautifulSoup` can also be helpful for working with tabular data. We don't always need to create a DataFrame with all of the table contents. Sometimes we just want to extract some data from a table. 

Let's experiment with this using the publications page of the CS Ed Research Group's site:
https://uoftcsed.github.io/pubs/



In [ ]:
url = "https://uoftcsed.github.io/pubs/"

response = requests.get(url)
response.encoding = response.apparent_encoding

if response.status_code == 200:
    html = response.text
else:
    print(f'The request was not successful: status code {response.status_code}')

soup = BeautifulSoup(html, 'html.parser')

# We can tables and their rows.
table = soup.find('table')
rows = table.find_all('tr')

# For this webpage, the cells (<td>) have class attributes associated with
# them. The first column has class='bibline' and the second class='bibref'.

# We can get the cells for bibitem class.
cells = table.find_all('td', class_='bibitem')

authors = []

for cell in cells:
    citation = cell.get_text()

    # We can split on the opening quote for the paper title. The first item
    # of the list generated would contain the authors.
    authors.append(citation.split('"')[0])

print(authors)

['Sophia Krause-Levy, Andrew Petersen, Oladele O. Campbell, William G. Griswold, Leo Porter, Oluwatoyin Adelakun-Adeyemo, Jennifer Campbell, Michelle Craig, Adrienne Decker, Sebastian Dziallas, Carrie Demmans Epp, David R. Gibson, Yekaterina Kharitonova, Devorah Kletenik, David L. Largent, Emma McDonald, Brian M. McSkimming, Tina L. Peterson, Caroline Sih, Cynthia Taylor and Neena Thota.  ', 'Rita Garcia and Michelle Craig.  ', 'Marina Tawfik, Andrew Petersen, Leo Portero and Lisa Zhang.  ', 'Naaz Sibia, Jessica Wen, Amber Richardson, Yashika Jain, Angela Zavaleta Bernuy, Bogdan Simion, Andrew Petersen, Carolina Nobre and Michael Liut.  ', 'Sadia Sharmin and Yohan Kim.  ', 'Caroline Pechenik, Angela Zavaleta Bernuy, Selina Marianna Shah, Shirley de Wit, Emmanuel Awuni Kolog, Oscar Karnalim, Mohammed Farghally, Carlos Anibal Suárez, Jack Parkinson, Leo Porter, Rodrigo Duran, Paul Vrbik, Brian Harrington, Lisa Zhang, Michael Liut and Andrew Petersen.  ', 'Khushi Malik, Amber Richardson, 